# HPC setup without a terminal (NYU Cloud Bursting)

Run this notebook in an **Open OnDemand → Interactive Apps → Jupyter** session (account `cs_gy_6643-2026fa`, a CPU partition such as `n2c48m24`, 4 CPUs, ~16 GB, 3 hours). It runs on a compute node, so installs are allowed. Every cell is a shell command run for you; no terminal is needed.

Before this: with the **Files** app, create `/scratch/<NetID>/penalty_kick/` with subfolders `data`, `code`, `code/jobs`, `env`, `logs`, `outputs`, `staging`; upload the ZIP to `data/`; upload `Penalty_Kick_HPC.ipynb`, `run_penalty_kick.py`, `penalty_kick.sbatch` and this notebook to `code/`; upload the `jobs/*.json` files to `code/jobs/`.

Run the cells in order. Cells 3 and 4 run only once.

In [ ]:
# 1) Settings for this setup notebook (edit IMAGE after cell 2 prints the available images)
import os
NETID = os.environ['USER']
BASE = f'/scratch/{NETID}/penalty_kick'
IMAGE = ''          # e.g. '/share/apps/images/cuda12.3.2-cudnn9.0.0-ubuntu-22.04.4.sif'
TIME_LIMIT = ''     # e.g. '12:00:00' (from the partition limits printed by cell 2)
os.environ.update(BASE=BASE, IMAGE=IMAGE, OVERLAY=f'{BASE}/env/overlay-penalty.ext3')
print('Base folder:', BASE)

In [ ]:
%%bash
# 2) What is available: tools, CUDA images, partition time limits, and whether the uploads are visible here
echo "== tools =="; which apptainer sbatch squeue sinfo 2>&1
echo "== CUDA images (pick one for IMAGE in cell 1) =="; ls /share/apps/images/ | grep -i cuda
echo "== partition time limits =="; sinfo -o "%P %l" 2>&1 | head -20
echo "== your uploads =="; ls -lh $BASE/data $BASE/code $BASE/code/jobs 2>&1

In [ ]:
%%bash
# 3) ONE TIME: create the 15 GB overlay that will hold Python and all packages
set -e
mkdir -p $BASE/env $BASE/logs $BASE/outputs $BASE/staging
cd $BASE/env
if [ -f overlay-penalty.ext3 ]; then echo "Overlay already exists"; else
  cp -rp /share/apps/overlay-fs-ext3/overlay-15GB-500K.ext3.gz . && gunzip overlay-15GB-500K.ext3.gz
  mv overlay-15GB-500K.ext3 overlay-penalty.ext3 && echo "Overlay created"; fi

In [ ]:
%%bash
# 4) ONE TIME (10-30 min): install Miniforge and the pinned packages inside the overlay
set -e
[ -n "$IMAGE" ] || { echo "Set IMAGE in cell 1 and rerun it first"; exit 1; }
apptainer exec --fakeroot --overlay $OVERLAY:rw $IMAGE /bin/bash -c '
set -e
cd /tmp
if [ ! -d /ext3/miniforge3 ]; then
  wget -q --no-check-certificate https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
  bash Miniforge3-Linux-x86_64.sh -b -p /ext3/miniforge3
fi
printf "#!/bin/bash\nunset -f which\nsource /ext3/miniforge3/etc/profile.d/conda.sh\nexport PATH=/ext3/miniforge3/bin:\$PATH\nexport PYTHONPATH=/ext3/miniforge3/bin:\$PATH\n" > /ext3/env.sh
source /ext3/env.sh
pip install -q torch torchvision
pip install -q numpy==2.2.6 "scipy>=1.15,<1.17" "pandas>=2.2.3,<3" scikit-learn==1.7.2 av==16.1.0 catboost==1.2.8 \
  transformers==5.3.0 "huggingface-hub>=1.32,<2" ultralytics==8.4.0 opencv-python-headless tqdm \
  "timm>=1.0.15,<2" "einops>=0.8,<1" jupyter ipykernel packaging matplotlib
python -c "import torch, transformers, catboost, ultralytics, av, cv2; print(\"torch\", torch.__version__, \"transformers\", transformers.__version__)"
'
echo "Environment ready."

In [ ]:
# 5) Write the time limit and image into penalty_kick.sbatch (replaces the UNVERIFIED placeholders)
import re, pathlib
assert IMAGE and TIME_LIMIT, 'Set IMAGE and TIME_LIMIT in cell 1 and rerun it'
p = pathlib.Path(BASE, 'code', 'penalty_kick.sbatch'); s = p.read_text()
s = re.sub(r'^#SBATCH --time=\S+', f'#SBATCH --time={TIME_LIMIT}', s, flags=re.M)
s = re.sub(r'^IMAGE=.*$', f'IMAGE=${{PENALTY_IMAGE:-{IMAGE}}}', s, flags=re.M)
p.write_text(s)
print('\n'.join(l for l in s.splitlines() if l.startswith(('#SBATCH --time', 'IMAGE=', 'OVERLAY='))))

In [ ]:
%%bash
# 6) Submit the SMOKE test (a few minutes). If sbatch is not found here, use Jobs > Job Composer instead (see README cell).
cd $BASE/code && sbatch penalty_kick.sbatch jobs/smoke.json

In [ ]:
%%bash
# 7) Status and the end of the newest log (rerun this cell to refresh)
squeue -u $USER
LOG=$(ls -t $BASE/logs/*.out 2>/dev/null | head -1); echo "== $LOG =="; [ -n "$LOG" ] && tail -n 40 "$LOG"

In [ ]:
%%bash
# 8) After the smoke log shows "END-TO-END SMOKE TEST: PASS": submit the FULL run (VideoMAE, nested_cv, submission)
cd $BASE/code && sbatch penalty_kick.sbatch jobs/baseline_nested.json

In [ ]:
%%bash
# 9) Where results appear (submission.csv once all folds finish)
ls -d $BASE/outputs/runs/hpc/*/ 2>/dev/null
ls $BASE/outputs/runs/hpc/*/folds/*/COMPLETE 2>/dev/null
ls -lh $BASE/outputs/runs/hpc/*/submission.csv 2>/dev/null || echo "No submission yet"

## If `sbatch` is not available inside Jupyter: Job Composer
1. With the **Files** app, copy `code/jobs/smoke.json` to `code/job.json` (later: copy `code/jobs/baseline_nested.json` over `code/job.json`).
2. **Jobs → Job Composer → New Job → From Specified Path**: path `/scratch/<NetID>/penalty_kick/code`, script name `penalty_kick.sbatch`. Save, select the job and click **Submit**. The script reads `job.json` from that folder when no argument is given.
3. **Jobs → Active Jobs** shows status (and a delete button to cancel). Logs are in `penalty_kick/logs/` in the Files app; refresh to see new lines.